# How to Use

1. Run everything in the **Setup** section. 
    - Make sure your dataset path is set correctly by following the `# TODO` comment.
    - Some requirements for the datasets:
        - The data must be on the **first sheet** in the Excel document.
        - The **first row** must be the column names. 
        - The test won't run if the Excel file is open

2. Understand and Prepare Tests
    - Review what each test does before running it:
        - For a quick summary, refer to this [table](../REFERENCE_TABLE.md)
        - For detailed examples, refer to this [document](https://086gc.sharepoint.com/:w:/r/sites/PacificSalmonTeam/Shared%20Documents/General/02%20-%20PSSI%20Secretariat%20Teams/04%20-%20Strategic%20Salmon%20Data%20Policy%20and%20Analytics/02%20-%20Data%20Governance/00%20-%20Projects/10%20-%20Data%20Quality/Reference%20Documents/Data%20Quality%20Framework%20Documentation.docx?d=w6e45b6f006b74a15b4ce44a59a49ae19&csf=1&web=1&e=bsaWFi).
    - *Define* or *update* parameters for each test in the `test_params` dictionary.
    - **Optional**: adjust weights for dimension scoring (by default, all tests and dimensions are weighted equally; weights must add up to 1)

3. After running all the tests, the Excel document for logging the scores can be uploaded to Sharepoint using the function "Saving the file to sharepoint".

**Note:** The Output Reports are used for when a data steward is asking about why their dataset gets a certain score. If the test is not in Output Reports, then running the test itself will generate an output that can be put into a report.  

# Setup

Please run everything in the set up, and double check the path to your dataset is correct.

All of these functions are used in the process of calculating data quality. 

In [1]:
import sys, os
REPO_ROOT = os.path.abspath("..")
sys.path.append(REPO_ROOT) # allows notebook in run_tests to import from folders in parent folder

In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os
import re
from difflib import SequenceMatcher
from datetime import datetime
import nbformat
import gc
import os

# Import dimensions
from dimensions.accessibility.dimension_reference import Accessibility
from dimensions.accuracy.dimension_reference import Accuracy
from dimensions.completeness.dimension_reference import Completeness
from dimensions.consistency.dimension_reference import Consistency
from dimensions.interdependency.dimension_reference import Interdependency
from dimensions.relevance.dimension_reference import Relevance
from dimensions.timeliness.dimension_reference import Timeliness
from dimensions.uniqueness.dimension_reference import Uniqueness
from utils.core_operations import calculate_dimension_score, calculate_DQ_grade

In [3]:
gc.collect()

79

Make sure to set the full path to your dataset below

In [4]:
# TODO: Enter the full path to your dataset
DATA_FILE_PATH = f"C:/Users/username/OneDrive - DFO-MPO/dataset.xlsx" # Full path to csv or xlsx dataset

LOGGING_PATH = os.path.join(REPO_ROOT, "run_tests", "DQS_Output_Log_Test.xlsx") # Where detailed test outputs are stored if dimension has return_type set to 'dataset'
DIMENSION_SCORES = [] # Stores the final score for each dimension used to calculate final grade

In [4]:
# TODO: Enter the full path to your dataset
DATA_FILE_PATH = f"C:/Users/liman/OneDrive - DFO-MPO/Dummy Data Testing.xlsx" # Full path to csv or xlsx dataset

LOGGING_PATH = os.path.join(REPO_ROOT, "run_tests", "DQS_Output_Log_Test.xlsx") # Where detailed test outputs are stored if dimension has return_type set to 'dataset'
DIMENSION_SCORES = [] # Stores the final score for each dimension used to calculate final grade

# Data Quality Tests

### Accessibility

**Accessibility Type 1 (S1)**

Gives a score based on whether a metadata file exists for the specified dataset.

#### Prepare Accessibility Tests

In [5]:
accessibility_tests = Accessibility(
    dataset_path=DATA_FILE_PATH,
    return_type="dataset", # Can set to score or dataset (if dataset will create 1 sentence summary in output report)
    logging_path = LOGGING_PATH, # By default this is stored in memory (if set and return_type is dataset, will store csv of all numbers used in test calculations)
    test_params={
        "S1": {"s1_has_metadata": True}
    }
)

#### Run Accessibility Tests

In [6]:
# Individual test scores
scores = accessibility_tests.run_tests()
scores

[{'test': 'S1', 'value': 1}]

In [7]:
# Accessibility dimension score
accessibility_score = calculate_dimension_score("Accessibility", scores=scores, weights={})
DIMENSION_SCORES.append(accessibility_score)
print(accessibility_score)

{'dimension': 'Accessibility', 'score': 1.0}


### Accuracy

**Accuracy Type 1 (A1)**

Checks whether letters or symbols exist in the specified numeric column(s).

**Accuracy Type 2 (A2)**

Detects outliers in the specified numeric column(s) using the interquartile range (IQR) method.

**Accuracy Type 3 (A3)**

Checks whether values in the specified aggregated column (e.g., Total) equal the sum of their respective component columns.

**Accuracy Type 4 (A4)**

Checks whether related timestamp columns are in chronological order, with the first specified column in each pair assumed to be the start timestamp.

#### Prepare Accuracy Tests

In [8]:
# Using default threshold and minimum score for A2 
accuracy_tests = Accuracy(
    dataset_path=DATA_FILE_PATH,
    return_type="dataset", # Can set to score or dataset (if dataset will create 1 sentence summary in output report)
    logging_path = LOGGING_PATH, # By default this is stored in memory (if set and return_type is dataset, will store csv of all numbers used in test calculations)
    test_params={
        "A1": {"a1_column_names": ['AREA']},
        "A2": {"a2_column_names": ['TOTAL_RETURN_TO_RIVER'], 
               "a2_groupby_column": ['SPECIES'], 
               "a2_threshold": 1.5, 
               "a2_minimum_score": 0.85},
        "A3": {"a3_column_names": ['NATURAL_ADULT_SPAWNERS', 'NATURAL_JACK_SPAWNERS'], 
               "a3_agg_column": ['NATURAL_SPAWNERS_TOTAL']},
        "A4": {"a4_column_pairs": [('START_DTT', 'END_DTT')]}
    }
)

#### Run Accuracy Tests

In [9]:
# Individual test scores
scores = accuracy_tests.run_tests()
scores

[{'test': 'A1', 'value': np.float64(0.6363636363636364)},
 {'test': 'A2', 'value': np.float64(1.0)},
 {'test': 'A3', 'value': np.float64(0.7272727272727273)},
 {'test': 'A4', 'value': np.float64(0.8181818181818181)}]

In [10]:
# Accuracy dimension score
accuracy_score = calculate_dimension_score("Accuracy", scores=scores, weights={})
DIMENSION_SCORES.append(accuracy_score)
print(accuracy_score)

{'dimension': 'Accuracy', 'score': np.float64(0.7954545454545454)}


### Completeness

**Completeness Type 1 (P1)**

Checks whether there are blanks or NULL values in the entire specified dataset.

**Completeness Type 2 (P2)**

Finds column pairs whose missing values are correlated above a specified threshold, exploring whether missingness occurs independently or follows consistent patterns.

#### Prepare Completeness Tests

In [11]:
# Using default thresholds for both tests
completeness_tests = Completeness(
    dataset_path=DATA_FILE_PATH,
    return_type="dataset", # Can set to score or dataset (if dataset will create 1 sentence summary in output report)
    logging_path = LOGGING_PATH, # By default this is stored in memory (if set and return_type is dataset, will store csv of all numbers used in test calculations)
    test_params={
        "P1": {"p1_exclude_columns": ['Contact'], 
               "p1_threshold": 0.75},
        "P2": {"p2_threshold": 0.50}
    }
)

C:\Users\liman\AppData\Local\Programs\Python\Python313\Lib\site-packages\dython\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution


#### Run Completeness Tests

In [12]:
# Individual test scores
scores = completeness_tests.run_tests()
scores

[{'test': 'P1', 'value': np.float64(0.7520661157024794)},
 {'test': 'P2', 'value': 0.7142857142857143}]

In [13]:
# Completeness dimension score
completeness_score = calculate_dimension_score("Completeness", scores=scores, weights={})
DIMENSION_SCORES.append(completeness_score)
print(completeness_score)

{'dimension': 'Completeness', 'score': np.float64(0.7331759149940968)}


### Consistency

**Consistency Type 1 (C1)**

Detects near-duplicate entries in the specified column(s) that likely refer to the same entity despite minor differences in spelling or naming conventions.

**Consistency Type 2 (C2)**

Checks whether string values in the specified column(s) consistently follow naming conventions found in the specified reference dataset.

**Consistency Type 3 (C3)**

Checks whether string values in the specified column(s) are similar to official province or territory names using the Levenshtein similarity ratio.

**Consistency Type 4 (C4)**

Checks whether date-time values in the specified column(s) follow a specified format, such as ISO 8601.

**Consistency Type 5 (C5)**

Checks whether geographic coordinates in the specified column(s) follow Decimal Degrees (DD) format and represent valid latitude and longitude values, optionally restricting validation to DFO’s Pacific Region.

#### Prepare Consistency Tests

In [14]:
# C2 mapping of dataset columns to reference dataset columns for comparison
# Format: "dataset column" : "reference column"
column_mapping = {
    "STOCK_CU_NAME": "CU_Display",
    "STOCK_CU_INDEX": "FULL_CU_IN",
}

# C2 full path to csv or xlsx reference dataset
REF_FILE_PATH = "C:/Users/username/OneDrive - DFO-MPO/reference_dataset.xlsx"

In [14]:
# C2 mapping of dataset columns to reference dataset columns for comparison
# Format: "dataset column" : "reference column"
column_mapping = {
    "STOCK_CU_NAME": "CU_Display",
    "STOCK_CU_INDEX": "FULL_CU_IN",
}

# C2 full path to csv or xlsx reference dataset
REF_FILE_PATH = "C:/Users/liman/OneDrive - DFO-MPO/Pacific Salmon Population Unit Crosswalk_Final_20240513.xlsx"

In [15]:
# Using default thresholds, stop words, format, and region for all tests
consistency_tests = Consistency(
    dataset_path=DATA_FILE_PATH,
    return_type="dataset", # Can set to score or dataset (if dataset will create 1 sentence summary in output report)
    logging_path = LOGGING_PATH, # By default this is stored in memory (if set and return_type is dataset, will store csv of all numbers used in test calculations)
    test_params={
        "C1": {"c1_column_names": ['PROJ_NAME'], 
               "c1_threshold": 0.91, 
               "c1_stop_words": ["the", "and"]},
        "C2": {"c2_column_mapping": column_mapping}, 
               "c2_threshold": 1, 
               "c2_stop_words": ["activity"], 
               "ref_dataset_path": REF_FILE_PATH,
        "C3": {"c3_column_names": ['PROVINCE','PROVINCE_OTHER'],
               "c3_threshold": 0.91},
        "C4": {"c4_column_names": ['DATE_1', 'DATE_2'],
               "c4_format":'%Y-%m-%d %H:%M:%S'},
        "C5": {"c5_column_names": ['STOCK_LATITUDE', 'STOCK_LONGITUDE'], 
               "c5_region": "All"}
    }
)

#### Run Consistency Tests

In [16]:
# Individual test scores
scores = consistency_tests.run_tests()
scores

[{'test': 'C1', 'value': np.float64(1.0)},
 {'test': 'C2', 'value': 1.0},
 {'test': 'C3', 'value': 0.8636363636363636},
 {'test': 'C4', 'value': np.float64(0.8535353535353535)},
 {'test': 'C5', 'value': np.float64(0.7727272727272727)}]

In [17]:
# Consistency dimension score
consistency_score = calculate_dimension_score("Consistency", scores=scores, weights={})
DIMENSION_SCORES.append(consistency_score)
print(consistency_score)

{'dimension': 'Consistency', 'score': np.float64(0.897979797979798)}


### Interdependency

**Interdependency Type 1 (I1)**

Identifies proxy variables by finding non-sensitive features whose correlation with the specified sensitive feature(s) exceed a specified threshold.

#### Prepare Interdependency Tests

In [18]:
# Using default thresholds for this test
interdependency_tests = Interdependency(
    dataset_path=DATA_FILE_PATH,
    return_type="dataset", # Can set to score or dataset (if dataset will create 1 sentence summary in output report)
    logging_path = LOGGING_PATH, # By default this is stored in memory (if set and return_type is dataset, will store csv of all numbers used in test calculations)
    test_params={
        "I1": {"i1_sensitive_columns": ['Contact', 'CU_ID', 'CU_Name'], 
               "i1_threshold": 0.75}
    }
)

#### Run Interdependency Tests

In [19]:
# Individual test scores
scores = interdependency_tests.run_tests()
scores

[{'test': 'I1', 'value': 0.18518518518518523}]

In [20]:
# Interdependency dimension score
interdependency_score = calculate_dimension_score("Interdependency", scores=scores, weights={})
DIMENSION_SCORES.append(interdependency_score)
print(interdependency_score)

{'dimension': 'Interdependency', 'score': 0.18518518518518523}


## Relevance

**Relevance Type 1 (R1)**

Currently an empty template test.

## Timeliness

**Timeliness Type 1 (T1)**

Currently an empty template test.

## Uniqueness

**Uniqueness Type 1 (U1)**

Detects duplicate rows across the entire specified dataset.

#### Prepare Uniqueness Tests

In [21]:
# Using default thresholds for this test
uniqueness_tests = Uniqueness(
    dataset_path=DATA_FILE_PATH,
    return_type="dataset", # Can set to score or dataset (if dataset will create 1 sentence summary in output report)
    logging_path = LOGGING_PATH, # By default this is stored in memory (if set and return_type is dataset, will store csv of all numbers used in test calculations)
)

#### Run Uniqueness Tests

In [22]:
scores = uniqueness_tests.run_tests()
scores

Duplicate Rows:
   PROGRAM_CODE PROJ_NAME SPECIES_NAME RUN_NAME  BROOD_YEAR  STOCK_NAME  \
0           AFS  Emily Lk      Sockeye   Summer        2001  Tankeeah R   
10          AFS  Emily Lk      Sockeye   Summer        2001  Tankeeah R   

   STOCK_PROD_AREA_CODE  STOCK_GFE_ID  STOCK_GFE_NAME  STOCK_POP_ID  ...  \
0                  CCST          1001  TANKEEAH RIVER       51935.0  ...   
10                 CCST          1001  TANKEEAH RIVER       51935.0  ...   

   TOTAL_BROODSTOCK_REMOVALS  OTHER_REMOVALS TOTAL_RETURN_TO_RIVER  \
0                        0.0             NaN                   NaN   
10                       0.0             NaN                   NaN   

   ENUMERATION_METHODS  ADULT_PRESENCE JACK_PRESENCE  START_DTT    END_DTT  \
0                  NaN         PRESENT       PRESENT 2023-07-12 2023-10-04   
10                 NaN         PRESENT       PRESENT 2023-07-12 2023-10-04   

                 DATE_1               DATE_2  
0   2023-07-12 00:00:00  2021-03-04 

[{'test': 'U1', 'value': 0.8181818181818181}]

In [23]:
# Uniqueness dimension score
uniqueness_score = calculate_dimension_score("Uniqueness", scores=scores, weights={})
DIMENSION_SCORES.append(uniqueness_score)
print(uniqueness_score)

{'dimension': 'Uniqueness', 'score': 0.8181818181818181}


## Determine Overall Data Quality

In [24]:
# Caclulate data quality
print(f'Calculate Data Quality for this dataset is: {calculate_DQ_grade(DIMENSION_SCORES)}')

Calculate Data Quality for this dataset is: Good
